# x8_chuva — Precipitação Mensal Acumulada CHIRPS v2

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** precipitação acumulada (CHIRPS), mm, municipal, mensal  
**Conceito e Mecanismo Econômico:** Volume mensal de chuva (mm) derivado de estimativas satelitais calibradas com pluviômetros. Informa o regime hídrico municipal.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h2__precip_mm__chirps_v2__mensal_candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Explorar precip_mm; definir janela produtiva/decisória antes da agregação anual ou por safra.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município (141 municípios).
- **`ano`**: Ano civil.
- **`mes`**: Mês (1 a 12).
- **`precip_mm`**: Chuva acumulada no mês (milímetros).
- **`fracao_pixels_validos`**: Acurácia espacial dos pixels válidos.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h2__precip_mm__chirps_v2__mensal_candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x8_chuva`:
- **Atenção:** Explorar precip_mm; definir janela produtiva/decisória antes da agregação anual ou por safra.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['mes'] = pd.to_numeric(df['mes'], errors='coerce')
df['precip_mm'] = pd.to_numeric(df['precip_mm'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# Aglutinar Nova Ubiratã pela média de precipitação
df = df.groupby(['codigo_ibge', 'ano', 'mes'], as_index=False)['precip_mm'].mean()
print(f"Municípios únicos: {df['codigo_ibge'].nunique()} (141 municípios)")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da Precipitação Mensal (mm):")
print(df['precip_mm'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['precip_mm'], bins=30, kde=True, ax=axes[0], color='deepskyblue')
axes[0].set_title("Distribuição da Precipitação Mensal (mm)")

sazonal = df.groupby('mes')['precip_mm'].mean().reset_index()
sns.barplot(data=sazonal, x='mes', y='precip_mm', ax=axes[1], color='dodgerblue')
axes[1].set_title("Ciclo Sazonal Médio de Chuva em MT (mm)")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
anual = df_painel.groupby(['ano', 'codigo_ibge'])['precip_mm'].sum().reset_index()
sns.lineplot(data=anual, x='ano', y='precip_mm', estimator='mean', color='steelblue')
plt.title("Precipitação Anual Média em MT (2001-2024)")
plt.ylabel("Chuva Acumulada Anual (mm)")
plt.show()

## Fase 6: Feature Engineering — Precipitação e Janelas Agronômicas Críticas

### 1. Step 1: Agrupamento em Janelas Produtivas Agronômicas
- **Por que a soma do ano civil (Janeiro a Dezembro) é insuficiente:** Uma tempestade em julho (no meio da estação seca) não ajuda a lavoura de soja. O agricultor depende de chuva em momentos específicos do ciclo biológico.
- **Três Métricas Anualizadas Construídas:**
  1. **Chuva Total Anual:** Precipitação acumulada de 12 meses (mm).
  2. **Safra de Verão (Outubro $t-1$ a Março $t$):** Período crítico que define a semeadura, floração e enchimento de grãos da soja.
  3. **Segunda Safra / Safrinha (Fevereiro a Maio $t$):** Período em que a chuva vai diminuindo e define o sucesso do milho safrinha.

### 2. Step 2: Defasagens e Estresse Hídrico
- Defasagem $t-1$ da chuva da safra de verão para capturar o impacto financeiro da safra anterior na decisão de conversão presente.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Agregação por Ano Calendário (Janeiro a Dezembro)
precip_anual = df.groupby(['codigo_ibge', 'ano'])['precip_mm'].sum().reset_index(name='precip_total_anual_mm')

# 2. Janela Agronômica da Safra de Verão (Outubro a Março)
# Outubro a Dezembro pertencem ao ciclo de plantio que colhe no início do ano seguinte
df_copia = df.copy()
df_copia['safra_ano'] = np.where(df_copia['mes'] >= 10, df_copia['ano'] + 1, df_copia['ano'])
safra_verao = df_copia[df_copia['mes'].isin([10, 11, 12, 1, 2, 3])].groupby(['codigo_ibge', 'safra_ano'])['precip_mm'].sum().reset_index(name='precip_safra_verao_mm')
safra_verao.rename(columns={'safra_ano': 'ano'}, inplace=True)

# 3. Janela da Safrinha de Milho (Fevereiro a Maio)
safrinha = df[df['mes'].isin([2, 3, 4, 5])].groupby(['codigo_ibge', 'ano'])['precip_mm'].sum().reset_index(name='precip_safrinha_mm')

# 4. Consolidar Painel de Variáveis Climáticas
painel_chuva = pd.merge(precip_anual, safra_verao, on=['codigo_ibge', 'ano'], how='left')
painel_chuva = pd.merge(painel_chuva, safrinha, on=['codigo_ibge', 'ano'], how='left')

# Defasagem temporal de 1 ano (t-1)
painel_chuva.sort_values(['codigo_ibge', 'ano'], inplace=True)
painel_chuva['precip_safra_verao_lag1'] = painel_chuva.groupby('codigo_ibge')['precip_safra_verao_mm'].shift(1)

painel_chuva_final = painel_chuva[(painel_chuva['ano'] >= 2001) & (painel_chuva['ano'] <= 2024)].copy()

print("Métricas Climáticas Sazonais por Janela Agronômica:")
print(painel_chuva_final[['codigo_ibge', 'ano', 'precip_total_anual_mm', 'precip_safra_verao_mm', 'precip_safrinha_mm', 'precip_safra_verao_lag1']].head())